# D-MTHD tweet benchmark: full run

Before running: Settings (right panel) -> Accelerator **GPU T4 x2**, Internet **On**; Add Input -> dataset `andrewmvd/cyberbullying-classification`.

Then **Save Version -> Save & Run All (Commit)** so it keeps running for up to 12 hours after you close the tab. Every stage skips work whose results already exist, so re-running a killed notebook resumes.

Split work across accounts by editing `STUDENTS` in the second cell, for example one account runs only `distilbert-base-uncased:distilbert`.

In [ ]:
import os, subprocess
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
if not os.path.exists("/kaggle/working/dmthd-p3"):
    # if the repository is private, attach the code zip as a dataset instead and unzip it here
    r = subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/dmthd-p3"])
    if r.returncode != 0:
        import glob, zipfile
        z = glob.glob("/kaggle/input/**/dmthd-p3-code.zip", recursive=True)
        assert z, "clone failed and no dmthd-p3-code.zip found among the inputs"
        zipfile.ZipFile(z[0]).extractall("/kaggle/working/dmthd-p3")
os.chdir("/kaggle/working/dmthd-p3")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print(open("README.md").read()[:400])

In [ ]:
import os
os.environ["ROOT"] = "/kaggle/working"
os.environ["PYTHONPATH"] = "src"
os.environ["GPU"] = "1"
os.environ["SEEDS"] = "1,2,3"
os.environ["TEACHERS"] = "bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony"
os.environ["STUDENTS"] = "google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert"
os.environ["MODES"] = "ft,skd,uniform,dmthd"
!python kaggle/run_tweets.py --stage all --raw /kaggle/input/cyberbullying-classification/cyberbullying_tweets.csv

In [ ]:
# Pack everything worth keeping so it can be downloaded from the notebook output
!cd /kaggle/working && tar czf dmthd_runs.tgz runs cache/tweets/meta.json data/tweets/report.json && ls -la dmthd_runs.tgz
!python -m dmthd.aggregate --runs /kaggle/working/runs/tweets